# 10 · Phiên nghe last.fm (ticket #55)

Biến 253 triệu lượt nghe last.fm (Music4All-Onion, chưa có khái niệm phiên) thành bảng phiên giống `sessions.parquet` của MPD, để dùng lại `consecutive_pairs`, `smoothness` và điểm mix học (#23) trên bộ thứ hai. Cách cắt ghi ở `src/data/lastfm.py` và `docs/data-contract.md` (mục Bộ phụ last.fm).

**Input** (Add Input): Kaggle Dataset `mixnext-onion` (output notebook 09).

**Output** (`/kaggle/working/out/lastfm/`):
- `sessions.parquet`: `session_id`, `user_id`, `split` (theo user), `pos`, `track_id`
- `lastfm_log.json`: phễu từng bước, độ nhạy theo khoảng nghỉ 15/30/60 phút, thống kê từng split

#56 thêm `tracks.parquet` vào cùng notebook này (Dataset tạo từ output notebook chỉ nhận version mới từ chính notebook đó).

Chỉ cần CPU, bật **Internet** để clone repo. Nếu thiếu RAM: đặt `SAMPLE_USERS` < 1 (tỉ lệ ghi vào log).

In [ ]:
# Bootstrap: lấy code từ GitHub. Khi #55 đã merge vào main thì đổi BRANCH = "main".
BRANCH = "b"
REPO = "/tmp/MusicRecSys"
!rm -rf {REPO} && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys {REPO}
import sys; sys.path.insert(0, REPO)

In [ ]:
from pathlib import Path
ONION = sorted(Path("/kaggle/input").rglob("listening_events.parquet"))[0].parent
OUT = Path("/kaggle/working/out/lastfm")
SAMPLE_USERS = 1.0   # giảm (ví dụ 0.5) nếu hết RAM
print("input:", ONION, sorted(p.name for p in ONION.iterdir()))

In [ ]:
import json, time
from src.data.lastfm import main

t0 = time.time()
log = main(["--onion", str(ONION), "--out", str(OUT), "--sample-users", str(SAMPLE_USERS)])
print(f"{time.time() - t0:.0f} giây")

In [ ]:
# Phễu, độ nhạy theo khoảng nghỉ, các split
import pandas as pd
print(pd.DataFrame(log["funnel"]).to_string(index=False))
print()
print(pd.DataFrame(log["sensitivity"]).T.rename_axis("gap (phút)").to_string())
print()
print(pd.DataFrame(log["splits"]).T.to_string())
sessions = pd.read_parquet(OUT / "sessions.parquet")
lengths = sessions.groupby("session_id").size()
print(f"\nđộ dài phiên: trung vị {lengths.median():.0f}, p90 {lengths.quantile(0.9):.0f}, tối đa {lengths.max()}")

## Đăng output

1. **Save Version → Save & Run All (Commit)**.
2. Tab Output → **New Dataset** tên **`mixnext-lastfm`**, **Visibility: Private**, License CC BY 4.0 (dữ liệu gốc Music4All-Onion).

## Kết quả (chưa chạy)

Dán vào đây: thời gian chạy, bảng phễu, bảng độ nhạy (gap 15/30/60), bảng split, độ dài phiên.